# Urban Heat Island (UHI) Analysis — Pleiku, Gia Lai, Vietnam

Self-contained Colab notebook: downloads Landsat 8/9 imagery, extracts **NDVI**, **Land Surface Temperature (LST)** and **Emissivity**, quantifies the **Surface UHI intensity** (built-up vs. vegetated temperature gap), and renders an interactive **Folium webmap**.

Built on top of the [`uhi_detector`](https://github.com/ArchiColab/uhi_detector) repo's processing functions (`src/utils.py`), reused here without the building-segmentation model (not needed for this analysis).

Pleiku sits on the Central Highlands plateau (~13.98°N, 108.00°E, elevation ~780 m) — a rapidly urbanising city, which makes it a useful UHI case study outside the temperate cities the original project focused on.

**Outputs produced by this notebook:**
- Inline plots of NDVI / LST / Emissivity and a vegetated-vs-built-up temperature comparison
- A static PNG figure you can drop straight into a report or slide
- A standalone `.html` interactive webmap you can publish anywhere (GitHub Pages, a static file host, etc.) or download and open locally

## Prerequisites
- Free [USGS EarthExplorer account](https://ers.cr.usgs.gov/register) (needed once, to download Landsat scenes)
- Nothing else — no Google Drive mount or model files required


---
## 1. Setup — get the processing code

This clones the `uhi_detector` repo (for `src/utils.py`) when running on Colab. If you're running this notebook locally from inside the repo's `notebooks/` folder, it just uses the parent directory instead.

In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/ArchiColab/uhi_detector.git"

if IN_COLAB:
    REPO_DIR = Path("/content/uhi_detector")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    else:
        print("Repo already cloned.")
else:
    # Assume this notebook is running from <repo>/notebooks/
    REPO_DIR = Path("..").resolve()

if str(REPO_DIR) not in sys.path:
    sys.path.append(str(REPO_DIR))

print(f"Repo root: {REPO_DIR}")


In [ ]:
# Only the geospatial/UHI stack — no TensorFlow / segmentation dependencies needed.
%pip install -q rasterio geopandas folium pylandtemp pylandsat shapely


In [ ]:
import json
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import numpy as np
import rasterio

from src import utils

print("Imports OK.")


---
## 2. Configuration — area of interest & paths

In [ ]:
# ── Area of Interest ─────────────────────────────────────────────────────────
LOCATION_NAME = "pleiku"                 # used for file naming
CENTER_LAT    = 13.9833                  # Pleiku city centre
CENTER_LON    = 108.0000

# Bounding box for Landsat scene search (WGS84) — covers the greater Pleiku
# urban area plus surroundings (~30 x 28 km)
AOI_LAT_MIN, AOI_LON_MIN = 13.84, 107.86   # SW corner
AOI_LAT_MAX, AOI_LON_MAX = 14.12, 108.14   # NE corner

# ── Landsat search parameters ─────────────────────────────────────────────────
BEGIN_DATE = "2023-11-01"   # Dry season start (lower cloud cover)
END_DATE   = "2024-04-30"   # Dry season end
MAX_CLOUD  = 20             # Maximum cloud cover (%)

# NDVI threshold that separates vegetated from built-up/bare pixels for the
# UHI intensity comparison in Section 6. 0.3 is a common rule-of-thumb cutoff.
NDVI_THRESHOLD = 0.3


In [ ]:
# ── Directory layout ────────────────────────────────────────────────────────
dir_data        = REPO_DIR / "data"
dir_landsat     = dir_data / "landsat"
dir_source_base = dir_landsat / "source"     # raw downloaded scenes
dir_output_base = dir_landsat / "output"     # clipped/processed outputs
dir_geojson     = dir_data / "geojson"

# Working copy for this notebook's own outputs (plots, webmap)
dir_results = Path("results") if not IN_COLAB else Path("/content/uhi_pleiku_results")

for d in [dir_source_base, dir_output_base, dir_geojson, dir_results]:
    d.mkdir(parents=True, exist_ok=True)

geojson_path = dir_geojson / "pleiku.geojson"
print("Directories ready.")
print(f"Results will be saved to: {dir_results}")


---
## 3. Area of interest (AOI) polygon

Written out directly so the notebook works standalone even before the AOI file has been pushed to the repo. Edit the coordinates or re-draw the box at [geojson.io](https://geojson.io/) if you want a different extent.

In [ ]:
aoi_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {
                "name": "Pleiku, Gia Lai, Vietnam",
                "description": "Bounding box covering Pleiku urban area and surroundings (~30 x 28 km)",
                "city": "Pleiku",
                "province": "Gia Lai",
                "country": "Vietnam",
                "center_lat": CENTER_LAT,
                "center_lon": CENTER_LON,
            },
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [AOI_LON_MIN, AOI_LAT_MIN],
                    [AOI_LON_MIN, AOI_LAT_MAX],
                    [AOI_LON_MAX, AOI_LAT_MAX],
                    [AOI_LON_MAX, AOI_LAT_MIN],
                    [AOI_LON_MIN, AOI_LAT_MIN],
                ]],
            },
        }
    ],
}

with open(geojson_path, "w", encoding="utf-8") as f:
    json.dump(aoi_geojson, f)

print(f"AOI GeoJSON written to: {geojson_path}")


In [ ]:
coords = aoi_geojson["features"][0]["geometry"]["coordinates"][0]
lons = [c[0] for c in coords]
lats = [c[1] for c in coords]

fig, ax = plt.subplots(figsize=(5, 4))
ax.fill(lons, lats, alpha=0.2, color="green")
ax.plot(lons, lats, "g-")
ax.plot(CENTER_LON, CENTER_LAT, "ro", label="Pleiku centre")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Pleiku AOI")
ax.legend()
plt.tight_layout()
plt.show()


---
## 4. Download Landsat 8/9 imagery via `pylandsat`

[pylandsat](https://pypi.org/project/pylandsat/) authenticates with the **USGS M2M API** using your EarthExplorer credentials (free account at https://ers.cr.usgs.gov/register).

We use Landsat **Collection 2 Level-1** (`LANDSAT_OT_C2_L1`), which provides the raw top-of-atmosphere band values expected by the `pylandtemp` single-window LST algorithm used in `src/utils.py`.

> If you'd rather not enter credentials in the notebook, download the scene manually from https://earthexplorer.usgs.gov/, extract it into `data/landsat/source/<scene_id>/`, set `LANDSAT_ID` in **Section 5**, and skip straight there.

### Storing credentials

**On Colab:** click the key icon (🔑 Secrets) in the left sidebar → add two secrets, `USGS_USERNAME` and `USGS_PASSWORD` → toggle "Notebook access" on for both. The cell below reads them automatically and nothing gets written into the notebook file.

**Running locally:** call `Auth.store(username=..., password=...)` once — it saves to `~/.config/pylandsat/credentials` — then delete that line so the password isn't left in the notebook.

In [ ]:
from pylandsat import Auth, Catalog, Scene

if IN_COLAB:
    from google.colab import userdata
    try:
        Auth.store(username=userdata.get("USGS_USERNAME"), password=userdata.get("USGS_PASSWORD"))
        print("Credentials loaded from Colab Secrets.")
    except Exception as exc:
        print(f"Could not read Colab secrets automatically ({exc}).")
        print("Add USGS_USERNAME / USGS_PASSWORD in the Secrets panel, then re-run this cell.")
else:
    # Auth.store(username="YOUR_USGS_USERNAME", password="YOUR_USGS_PASSWORD")  # run once, then remove
    print("Assuming credentials are already stored in ~/.config/pylandsat/ (see markdown above).")


In [ ]:
# ── Search for available Landsat scenes covering Pleiku ───────────────────────
catalog = Catalog()

scenes = catalog.search(
    dataset="LANDSAT_OT_C2_L1",           # Landsat 8 & 9 Collection 2 Level-1
    ll=(AOI_LAT_MIN, AOI_LON_MIN),         # lower-left corner (lat, lon)
    ur=(AOI_LAT_MAX, AOI_LON_MAX),         # upper-right corner (lat, lon)
    begin=BEGIN_DATE,
    end=END_DATE,
    max_results=20,
    cloud_min=0,
    cloud_max=MAX_CLOUD,
)

print(f"Found {len(scenes)} scenes.\n")
for i, s in enumerate(scenes):
    print(f"[{i:02d}] {s.get('displayId', s.get('display_id', 'N/A'))}  "
          f"cloud={s.get('cloudCover', '?')}%  "
          f"date={s.get('acquisitionDate', s.get('acquisition_date', '?'))}")


In [ ]:
# ── Pick the scene index to process ────────────────────────────────────────────
SCENE_INDEX = 0    # <-- change to pick a different date/scene

selected = scenes[SCENE_INDEX]
LANDSAT_ID = selected.get("displayId") or selected.get("display_id")
print(f"Selected: {LANDSAT_ID}")
print(f"Cloud cover: {selected.get('cloudCover', selected.get('cloud_cover', '?'))}%")
print(f"Acquisition date: {selected.get('acquisitionDate', selected.get('acquisition_date', '?'))}")


In [ ]:
# ── Download the bands needed for NDVI / LST / Emissivity ─────────────────────
#   B4  Red   – NDVI + Emissivity + LST
#   B5  NIR   – NDVI + Emissivity + LST
#   B10 TIRS-1 – Land Surface Temperature (thermal)
BANDS_TO_DOWNLOAD = ["B4", "B5", "B10"]

scene = Scene(LANDSAT_ID)
scene.download(out_dir=str(dir_source_base), bands=BANDS_TO_DOWNLOAD)

print(f"Download complete. Files in: {dir_source_base / LANDSAT_ID}")


---
## 5. Clip bands to the Pleiku AOI

If you downloaded the scene manually, set `LANDSAT_ID` below to the extracted folder name first.

In [ ]:
# If you skipped Section 4 and downloaded manually, uncomment and set this:
# LANDSAT_ID = "LC08_L1TP_125052_20240115_20240115_02_T1"

dir_source = dir_source_base / LANDSAT_ID
dir_output = dir_output_base / LANDSAT_ID
dir_output.mkdir(parents=True, exist_ok=True)

band_4_path  = dir_source / f"{LANDSAT_ID}_B4.TIF"
band_5_path  = dir_source / f"{LANDSAT_ID}_B5.TIF"
band_10_path = dir_source / f"{LANDSAT_ID}_B10.TIF"

for p in [band_4_path, band_5_path, band_10_path]:
    status = "found" if p.exists() else "NOT FOUND"
    print(f"[{status}]  {p.name}")

print("\nClipping bands to Pleiku AOI ...")
band_4_clipped_path  = utils.clip_to_geojson(band_4_path,  geojson_path, dir_output)
band_5_clipped_path  = utils.clip_to_geojson(band_5_path,  geojson_path, dir_output)
band_10_clipped_path = utils.clip_to_geojson(band_10_path, geojson_path, dir_output)
print("Clipping complete.")

with rasterio.open(band_4_clipped_path) as src:
    print(f"Clipped shape: {src.read(1).shape}   CRS: {src.crs}   Resolution (m): {src.res}")


---
## 6. UHI analysis — NDVI, LST, Emissivity

Calculations use `src/utils.py`, which wraps the `pylandtemp` library.

| Metric | Formula | Bands |
|--------|---------|-------|
| **NDVI** | (B5 − B4) / (B5 + B4) | 4, 5 |
| **Emissivity** | from NDVI + B4 | 4, 5 |
| **LST** | Single-window algorithm (°C) | 4, 5, 10 |

In [ ]:
file_ndvi = dir_output / "ndvi.tif"
ndvi = utils.calc_ndvi(band_4_clipped_path, band_5_clipped_path, file_ndvi)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
plt.colorbar(im, ax=ax, label="NDVI")
ax.set_title(f"NDVI — Pleiku, Gia Lai\n{LANDSAT_ID}")
plt.tight_layout()
plt.show()

print(f"NDVI  min={ndvi.min():.3f}  max={ndvi.max():.3f}  mean={ndvi.mean():.3f}")


In [ ]:
file_lst = dir_output / "temperature_lst.tif"
lst = utils.calc_lst(band_4_clipped_path, band_5_clipped_path, band_10_clipped_path, file_lst)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(lst, cmap="RdBu_r")
plt.colorbar(im, ax=ax, label="LST (°C)")
ax.set_title(f"Land Surface Temperature — Pleiku\n{LANDSAT_ID}")
plt.tight_layout()
plt.show()

print(f"LST   min={lst.min():.1f}°C  max={lst.max():.1f}°C  mean={lst.mean():.1f}°C")


In [ ]:
file_emissivity = dir_output / "emissivity.tif"
emissivity = utils.calc_emissivity(band_4_clipped_path, band_5_clipped_path, file_emissivity)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(emissivity, cmap="YlOrRd")
plt.colorbar(im, ax=ax, label="Emissivity")
ax.set_title(f"Surface Emissivity — Pleiku\n{LANDSAT_ID}")
plt.tight_layout()
plt.show()


---
## 7. UHI intensity — vegetated vs. built-up temperature gap

This is the actual UHI measurement: using NDVI as a cheap proxy for land cover (no segmentation model needed), split pixels into **vegetated** vs. **built-up/bare** and compare their mean LST. The gap is the classic **Surface Urban Heat Island (SUHI) intensity** indicator.

In [ ]:
assert ndvi.shape == lst.shape, "NDVI and LST rasters must have matching shape for a pixel-wise comparison"

vegetation_mask = ndvi > NDVI_THRESHOLD
built_up_mask = ~vegetation_mask

mean_lst_vegetation = np.nanmean(lst[vegetation_mask])
mean_lst_built_up = np.nanmean(lst[built_up_mask])
uhi_intensity = mean_lst_built_up - mean_lst_vegetation

veg_pct = 100.0 * vegetation_mask.sum() / vegetation_mask.size

print(f"NDVI threshold                 : {NDVI_THRESHOLD}")
print(f"Vegetated area                 : {veg_pct:.1f}% of AOI")
print(f"Mean LST — vegetated pixels    : {mean_lst_vegetation:.1f} °C")
print(f"Mean LST — built-up/bare pixels: {mean_lst_built_up:.1f} °C")
print(f"Estimated Surface UHI intensity: {uhi_intensity:.1f} °C  (built-up minus vegetated)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart: mean LST by cover class
axes[0].bar(
    ["Vegetated\n(NDVI > %.1f)" % NDVI_THRESHOLD, "Built-up / bare\n(NDVI ≤ %.1f)" % NDVI_THRESHOLD],
    [mean_lst_vegetation, mean_lst_built_up],
    color=["#2e8b57", "#b22222"],
)
axes[0].set_ylabel("Mean LST (°C)")
axes[0].set_title(f"UHI intensity ≈ {uhi_intensity:.1f} °C")

# Scatter: NDVI vs LST (subsampled for speed/readability)
rng = np.random.default_rng(42)
flat_ndvi, flat_lst = ndvi.ravel(), lst.ravel()
valid = np.isfinite(flat_ndvi) & np.isfinite(flat_lst)
sample_idx = rng.choice(np.flatnonzero(valid), size=min(20000, valid.sum()), replace=False)

axes[1].scatter(flat_ndvi[sample_idx], flat_lst[sample_idx], s=2, alpha=0.2, color="#444")
axes[1].axvline(NDVI_THRESHOLD, color="black", linestyle="--", linewidth=1, label=f"threshold={NDVI_THRESHOLD}")
axes[1].set_xlabel("NDVI")
axes[1].set_ylabel("LST (°C)")
axes[1].set_title("NDVI vs. LST (per pixel)")
axes[1].legend()

fig.suptitle(f"Pleiku, Gia Lai — Surface UHI analysis — {LANDSAT_ID}", fontsize=12)
plt.tight_layout()

# ── Export as a static image (drop straight into a report/slide) ─────────────
uhi_summary_png = dir_results / f"{LOCATION_NAME}_uhi_summary.png"
fig.savefig(uhi_summary_png, dpi=200, bbox_inches="tight")
plt.show()

print(f"Static summary figure saved to: {uhi_summary_png}")


---
## 8. Reproject to WGS84 & create colour images

Needed to overlay the rasters on a web (Leaflet/Folium) map.

In [ ]:
file_ndvi_repr       = dir_output / "ndvi_reprojected.tif"
file_lst_repr        = dir_output / "temperature_lst_reprojected.tif"
file_emissivity_repr = dir_output / "emissivity_reprojected.tif"

reprojected_ndvi,       _ = utils.reproject_geotiff(file_ndvi,       file_ndvi_repr,       4326)
reprojected_lst,        _ = utils.reproject_geotiff(file_lst,        file_lst_repr,        4326)
reprojected_emissivity, _ = utils.reproject_geotiff(file_emissivity, file_emissivity_repr, 4326)

print("Reprojection to EPSG:4326 complete.")


In [ ]:
file_ndvi_colored       = dir_output / "ndvi_reprojected_colored.tif"
file_lst_colored        = dir_output / "temperature_lst_reprojected_colored.tif"
file_emissivity_colored = dir_output / "emissivity_reprojected_colored.tif"

colored_ndvi,       _ = utils.create_rgba_color_image(reprojected_ndvi.name,       file_ndvi_colored,       colormap="RdYlGn")
colored_lst,        _ = utils.create_rgba_color_image(reprojected_lst.name,        file_lst_colored,        colormap="RdBu_r")
colored_emissivity, _ = utils.create_rgba_color_image(reprojected_emissivity.name, file_emissivity_colored, colormap="YlOrRd")

print("RGBA colour images created.")


---
## 9. Interactive webmap (Folium) — export as HTML

In [ ]:
def load_colored_raster(path):
    """Read an RGBA GeoTIFF and return (array_HWC, bounds)."""
    with rasterio.open(path) as src:
        arr = np.moveaxis(src.read(), 0, 2)   # CHW -> HWC
        return arr, src.bounds

ndvi_arr,       ndvi_bounds       = load_colored_raster(file_ndvi_colored)
lst_arr,        lst_bounds        = load_colored_raster(file_lst_colored)
emissivity_arr, emissivity_bounds = load_colored_raster(file_emissivity_colored)

folium_map = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=12, tiles="CartoDB positron")

def add_overlay(m, arr, bounds, name, opacity=0.75, show=True):
    folium.raster_layers.ImageOverlay(
        image=arr,
        name=name,
        opacity=opacity,
        bounds=[[bounds.bottom, bounds.left], [bounds.top, bounds.right]],
        show=show,
    ).add_to(m)

add_overlay(folium_map, emissivity_arr, emissivity_bounds, "Emissivity", show=False)
add_overlay(folium_map, ndvi_arr,       ndvi_bounds,       "NDVI",       show=False)
add_overlay(folium_map, lst_arr,        lst_bounds,        "Land Surface Temperature (UHI)", show=True)

folium.GeoJson(
    aoi_geojson,
    name="AOI boundary",
    style_function=lambda _: {"fillOpacity": 0, "color": "black", "weight": 1.5, "dashArray": "4 4"},
).add_to(folium_map)

folium.Marker(
    [CENTER_LAT, CENTER_LON],
    tooltip="Pleiku city centre",
    icon=folium.Icon(color="red", icon="info-sign"),
).add_to(folium_map)

folium.LayerControl(collapsed=False).add_to(folium_map)

# ── Export as standalone HTML (publish this file directly, e.g. on GitHub Pages) ──
webmap_path = dir_results / f"{LOCATION_NAME}_uhi_webmap.html"
folium_map.save(str(webmap_path))
print(f"Webmap saved to: {webmap_path}")

folium_map


---
## 10. Summary & outputs

In [ ]:
print("Outputs for this run:")
print(f"  Landsat scene            : {LANDSAT_ID}")
print(f"  Surface UHI intensity    : {uhi_intensity:.1f} °C (built-up minus vegetated)")
print(f"  Static summary image     : {uhi_summary_png}")
print(f"  Interactive webmap (HTML): {webmap_path}")

if IN_COLAB:
    from google.colab import files
    print("\nUncomment the lines below to download the outputs to your machine:")
    print('# files.download(str(uhi_summary_png))')
    print('# files.download(str(webmap_path))')


### Suggested next steps

1. **Multi-date comparison:** download dry-season vs. wet-season scenes (change `BEGIN_DATE` / `END_DATE` in Section 2) and compare UHI intensity across seasons — Pleiku has a pronounced monsoon cycle, so the vegetation/LST relationship should shift noticeably.
2. **Publish the webmap:** the `.html` file from Section 9 is fully standalone — host it on GitHub Pages, Netlify, or any static file host, or just share the file directly.
3. **Sentinel-2 fusion:** Sentinel-2's 10 m Band 4 could sharpen the Landsat 30 m thermal map via pan-sharpening before computing LST, for a more detailed UHI map.